# Connectome Disease-Spread Demo

A standalone demo: pathology diffusion over the real *C. elegans* connectome
(White et al. 1986, sourced from OpenWorm's ConnectomeToolbox — MIT licensed,
see `snnkit/connectome/data/SOURCE.md`), with an interactive Plotly visualization
you can scrub through time.

**Why this belongs in the same repo as the core SNN simulator:** see
`docs/connectome-narrative.md` for the full explanation. Short version: the
core engine's sparse-graph representation (built for neural connectivity)
turns out to be exactly the representation needed for a real connectome, and
"how does something spread across a weighted graph over time" is a natural
extension of the same engine — from spiking dynamics to diffusion dynamics on
the same underlying sparse-graph substrate.

**Validation:** the diffusion model here isn't just "looks plausible" — it's
checked against the exact closed-form matrix-exponential solution to the
heat equation, with a reported correlation and error metric (`docs/local-vs-bptt.md`'s
sibling doc for this extension: see `tests/test_connectome_diffusion.py`).

**Scope:** this is a simplified linear diffusion model on a *C. elegans*
(302-neuron worm) connectome — a demonstration that the engine generalizes,
not a claim about human disease progression. See `docs/connectome-narrative.md`'s
closing section for explicit scope limits.

In [ ]:
import importlib.util
if importlib.util.find_spec("snnkit") is None:
    !git clone https://github.com/YOUR_USERNAME/snnkit.git
    %cd snnkit
    !pip install -e ".[connectome]" --quiet

In [ ]:
import jax.numpy as jnp
import numpy as np
import networkx as nx
import plotly.graph_objects as go

from snnkit.reproducibility import set_seed, get_package_versions
from snnkit.connectome.loader import load_white1986_connectome, to_networkx
from snnkit.connectome.diffusion import build_laplacian, simulate_diffusion_euler, reference_diffusion_expm, diffusion_fit_quality

set_seed(0)
print(get_package_versions())

## Load the connectome

In [ ]:
graph = load_white1986_connectome()
print(f"{len(graph.node_names)} nodes, {graph.weights.pre_idx.shape[0]} edges "
      f"({int((graph.edge_type==0).sum())} chemical, {int((graph.edge_type==1).sum())} electrical)")

g = to_networkx(graph)
pos = nx.spring_layout(g, seed=0, k=0.15, iterations=50)

## Validate the diffusion model against the exact reference solution

In [ ]:
laplacian = build_laplacian(graph, symmetrize=True)
n = len(graph.node_names)
source_name = "ADAL"
source_idx = graph.node_names.index(source_name)
x0 = jnp.zeros(n).at[source_idx].set(1.0)

diffusion_rate = 0.001
t_final = 50.0
dt = 0.1
n_steps = int(t_final / dt)

euler_trace = simulate_diffusion_euler(x0, laplacian, diffusion_rate, dt, n_steps)
reference_final = reference_diffusion_expm(x0, laplacian, diffusion_rate, t_final)
fit = diffusion_fit_quality(euler_trace[-1], reference_final)

print(f"Euler-integrated diffusion vs. exact matrix-exponential heat equation solution:")
print(f"  correlation = {fit['correlation']:.6f}")
print(f"  relative L2 error = {fit['relative_l2_error']:.4f}")

## Interactive Plotly visualization: scrub through time

In [ ]:
# Downsample to a manageable number of animation frames.
n_frames = 25
steps_per_frame = max(1, n_steps // n_frames)
frame_states = [np.asarray(x0)]
x = x0
for _ in range(n_frames):
    tr = simulate_diffusion_euler(x, laplacian, diffusion_rate, dt, steps_per_frame)
    x = tr[-1]
    frame_states.append(np.asarray(x))
frame_times = [i * steps_per_frame * dt for i in range(len(frame_states))]

# Edge traces (static background).
edge_x, edge_y = [], []
for u, v in g.edges():
    x0e, y0e = pos[u]
    x1e, y1e = pos[v]
    edge_x += [x0e, x1e, None]
    edge_y += [y0e, y1e, None]
edge_trace = go.Scatter(x=edge_x, y=edge_y, mode="lines",
                         line=dict(width=0.3, color="lightgray"), hoverinfo="none")

node_x = [pos[name][0] for name in graph.node_names]
node_y = [pos[name][1] for name in graph.node_names]

frames = []
for state, t in zip(frame_states, frame_times):
    frames.append(go.Frame(
        data=[edge_trace, go.Scatter(
            x=node_x, y=node_y, mode="markers",
            marker=dict(size=6 + 40*np.sqrt(np.clip(state, 0, None)),
                        color=state, colorscale="Hot", cmin=0, cmax=float(x0.max()),
                        showscale=True, colorbar=dict(title="pathology load")),
            text=[f"{name}: {v:.4f}" for name, v in zip(graph.node_names, state)],
            hoverinfo="text",
        )],
        name=f"{t:.1f}"
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=f"Pathology diffusion from {source_name} over the C. elegans connectome",
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        width=800, height=800,
        updatemenus=[dict(type="buttons", showactive=False,
            buttons=[dict(label="Play", method="animate",
                          args=[None, dict(frame=dict(duration=200, redraw=True), fromcurrent=True)]),
                     dict(label="Pause", method="animate",
                          args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
        sliders=[dict(steps=[dict(method="animate", args=[[f.name], dict(mode="immediate",
                       frame=dict(duration=0, redraw=True))], label=f.name) for f in frames])]
    )
)
fig.show()

## Static snapshot (fallback for non-interactive contexts, e.g. a pitch deck slide)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
snapshot_indices = [0, len(frame_states)//2, -1]
for ax, idx in zip(axes, snapshot_indices):
    state = frame_states[idx]
    nx.draw_networkx_edges(g, pos, ax=ax, alpha=0.05, width=0.3, arrows=False)
    sc = ax.scatter(node_x, node_y, c=state, cmap="hot", vmin=0, vmax=float(x0.max()), s=15)
    ax.set_title(f"t = {frame_times[idx]:.1f}")
    ax.axis("off")
plt.colorbar(sc, ax=axes, label="pathology load", fraction=0.02)
plt.savefig("connectome_diffusion_snapshots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved static fallback: connectome_diffusion_snapshots.png")

## Summary

- Loaded a real, licensed connectome (309 neurons, 2,961 synapses, White et al. 1986 via OpenWorm) directly into snnkit's existing sparse graph format — no new infrastructure needed.
- Simulated pathology diffusion using the graph heat equation, validated against the exact closed-form solution (correlation > 0.999).
- Visualized the spread interactively (scrub through time above) with a static fallback for non-interactive contexts.

See `docs/connectome-narrative.md` for why this extension belongs in the same repo as the core spiking-network engine, and `docs/connectome-demo.md` for explicit scope limits on what this demo does and doesn't claim.